# Simple Baseline

In [1]:

import pandas as pd
import numpy as np
from pathlib import Path
import json
import os
from chronos import Chronos2Pipeline
import time


LOAD_PATH = r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\inputs\clean\Consumption_dataset_1min_imputed.csv"
WEATHER_PATH = r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\inputs\clean\meteo_imputed.csv"
DATES_PATH = r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\inputs\clean\forecast_dates.json"

load_df = pd.read_csv(LOAD_PATH)
weather_df = pd.read_csv(WEATHER_PATH)

load_df.columns = load_df.columns.str.strip()
weather_df.columns = weather_df.columns.str.strip()

# adapt timestamps
load_df = load_df.rename(columns={"timestamp": "timestamp"})
weather_df = weather_df.rename(columns={"Date_Time": "timestamp"})

load_df["timestamp"] = pd.to_datetime(load_df["timestamp"])
weather_df["timestamp"] = pd.to_datetime(weather_df["timestamp"])

load_df = load_df.set_index("timestamp")
weather_df = weather_df.set_index("timestamp")

# 1-min load -> 5-min
load_5min = load_df.resample("5min").mean()

# weather is already 5-min, but this ensures alignment
weather_5min = weather_df.resample("5min").mean()

df = load_5min.join(weather_5min, how="inner").reset_index()

print(df.head())
print(df.columns.tolist())

WEATHER_COLS = [
    "Temp__C",
    "Hum__",
    "AvgWindSpeed_M_s",
    "Rain_Mm",
    "SolarRad_W_m_2",
]
TARGET_COL = "avg_power"
# ============================================================
# SETTINGS
# ============================================================
PRED_LEN = 288          # 24h forecast at 5-min resolution
CONTEXT_LEN = 288 * 30  # last 30 days as context, adjust if needed
QUANTILES = [0.1, 0.5, 0.9]

OUTPUT_DIR = Path(
    r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\chronos_5min"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with open(DATES_PATH, "r") as f:
    forecast_dates = json.load(f)

pipeline = Chronos2Pipeline.from_pretrained(
    "amazon/chronos-2",
    device_map="cpu",
)

# safety checks
df = df.sort_values("timestamp").reset_index(drop=True)
df = df.dropna(subset=[TARGET_COL] + WEATHER_COLS)

print(df[[TARGET_COL] + WEATHER_COLS].head())
print(df["timestamp"].min(), df["timestamp"].max())



def run_one_forecast(season, day_name, test_start):

    test_start = pd.Timestamp(test_start)
    test_end = test_start + pd.Timedelta(days=1)

    history_df = df[df["timestamp"] < test_start].tail(CONTEXT_LEN).copy()

    future_df = df[
        (df["timestamp"] >= test_start)
        & (df["timestamp"] < test_end)
    ].copy()

    if len(history_df) == 0:
        print(f"Skipping {season}-{day_name}: no history before {test_start}")
        return None

    if len(future_df) != PRED_LEN:
        print(f"Skipping {season}-{day_name}: expected {PRED_LEN}, got {len(future_df)}")
        return None

    # Historical target + historical weather only
    chronos_df = history_df[["timestamp", TARGET_COL] + WEATHER_COLS].copy()
    chronos_df = chronos_df.rename(columns={TARGET_COL: "target"})
    chronos_df["item_id"] = "building_consumption"

    chronos_df = chronos_df[
        ["item_id", "timestamp", "target"] + WEATHER_COLS
    ]

    pred_df = pipeline.predict_df(
        df=chronos_df,
        prediction_length=PRED_LEN,
        quantile_levels=QUANTILES,
        id_column="item_id",
        timestamp_column="timestamp",
        target="target",
        context_length=CONTEXT_LEN,
        freq="5min",
    )

    print(pred_df.head())
    print(pred_df.columns.tolist())

    # Chronos usually returns point forecast in "predictions"
    if "predictions" in pred_df.columns:
        pred_values = pred_df["predictions"].values[:PRED_LEN]
    elif "0.5" in pred_df.columns:
        pred_values = pred_df["0.5"].values[:PRED_LEN]
    elif 0.5 in pred_df.columns:
        pred_values = pred_df[0.5].values[:PRED_LEN]
    else:
        raise ValueError(f"Cannot find prediction column. Columns are: {pred_df.columns.tolist()}")

    out = future_df[["timestamp", TARGET_COL]].copy()
    out = out.rename(columns={TARGET_COL: "actual"})

    out["prediction"] = pred_values
    out["season"] = season
    out["day"] = day_name

    return out

all_results = []

for season, days in forecast_dates.items():
    for day_name, test_start in days.items():

        print("=" * 80)
        print(f"Running {season} - {day_name}: {test_start}")

        res = run_one_forecast(season, day_name, test_start)

        if res is not None:
            all_results.append(res)

            day_path = OUTPUT_DIR / f"chronos2_5min_SB_{season}_{day_name}.csv"
            res.to_csv(day_path, index=False)

            print(f"Saved: {day_path}")

results_df = pd.concat(all_results, ignore_index=True)

ModuleNotFoundError: No module named 'chronos'

# All features

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
import time
from chronos import Chronos2Pipeline

# ============================================================
# PATHS
# ============================================================
LOAD_PATH = r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\inputs\clean\Consumption_dataset_1min_imputed.csv"
WEATHER_PATH = r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\inputs\clean\meteo_imputed.csv"
DATES_PATH = r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\inputs\clean\forecast_dates.json"

OUTPUT_DIR = Path(
    r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\chronos_5min_engineered"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# SETTINGS
# ============================================================
TARGET_COL = "avg_power"

BASE_WEATHER_COLS = [
    "Temp__C",
    "Hum__",
    "AvgWindSpeed_M_s",
    "Rain_Mm",
    "SolarRad_W_m_2",
]

PRED_LEN = 288          # 24h @ 5-min
CONTEXT_LEN = 288 * 30  # 30 days
QUANTILES = [0.1, 0.5, 0.9]

# ============================================================
# LOAD DATA
# ============================================================
load_df = pd.read_csv(LOAD_PATH)
weather_df = pd.read_csv(WEATHER_PATH)

load_df.columns = load_df.columns.str.strip()
weather_df.columns = weather_df.columns.str.strip()

weather_df = weather_df.rename(columns={"Date_Time": "timestamp"})

load_df["timestamp"] = pd.to_datetime(load_df["timestamp"])
weather_df["timestamp"] = pd.to_datetime(weather_df["timestamp"])

load_df = load_df.set_index("timestamp")
weather_df = weather_df.set_index("timestamp")

# 1-min load -> 5-min
load_5min = load_df.resample("5min").mean()

# weather already 5-min, but resample for safety
weather_5min = weather_df.resample("5min").mean()

df = load_5min.join(weather_5min, how="inner").reset_index()
df = df.sort_values("timestamp").reset_index(drop=True)

# ============================================================
# ENGINEERED FEATURES
# ============================================================
def add_engineered_features(data, target_col):
    data = data.copy()

    ts = data["timestamp"]

    # ----------------------------
    # Time features
    # ----------------------------
    data["hour"] = ts.dt.hour
    data["minute"] = ts.dt.minute
    data["dayofweek"] = ts.dt.dayofweek
    data["dayofyear"] = ts.dt.dayofyear
    data["month"] = ts.dt.month
    data["is_weekend"] = (data["dayofweek"] >= 5).astype(int)

    # ----------------------------
    # Fourier features
    # ----------------------------
    steps_per_day = 288
    steps_per_week = 288 * 7

    step_of_day = data["hour"] * 12 + data["minute"] // 5
    step_of_week = data["dayofweek"] * steps_per_day + step_of_day

    for k in [1, 2, 3]:
        data[f"sin_day_{k}"] = np.sin(2 * np.pi * k * step_of_day / steps_per_day)
        data[f"cos_day_{k}"] = np.cos(2 * np.pi * k * step_of_day / steps_per_day)

    for k in [1, 2]:
        data[f"sin_week_{k}"] = np.sin(2 * np.pi * k * step_of_week / steps_per_week)
        data[f"cos_week_{k}"] = np.cos(2 * np.pi * k * step_of_week / steps_per_week)

    # ----------------------------
    # Load lag features
    # all are past-only
    # ----------------------------
    lag_steps = [
        1,        # 5 min ago
        2,        # 10 min ago
        3,        # 15 min ago
        6,        # 30 min ago
        12,       # 1 hour ago
        24,       # 2 hours ago
        288,      # same time yesterday
        288 * 2,  # same time 2 days ago
        288 * 7,  # same time last week
    ]

    for lag in lag_steps:
        data[f"{target_col}_lag_{lag}"] = data[target_col].shift(lag)

    # ----------------------------
    # Rolling historical load features
    # shifted by 1 to avoid using current value directly
    # ----------------------------
    rolling_windows = [
        3,     # 15 min
        12,    # 1 hour
        288,   # 1 day
        288*7  # 1 week
    ]

    for window in rolling_windows:
        shifted = data[target_col].shift(1)
        data[f"{target_col}_roll_mean_{window}"] = shifted.rolling(window).mean()
        data[f"{target_col}_roll_std_{window}"] = shifted.rolling(window).std()

    # ----------------------------
    # Weather-derived features
    # ----------------------------
    if "Temp__C" in data.columns:
        data["temp_sq"] = data["Temp__C"] ** 2
        data["heating_degree_18"] = np.maximum(18 - data["Temp__C"], 0)
        data["cooling_degree_22"] = np.maximum(data["Temp__C"] - 22, 0)

    if "SolarRad_W_m_2" in data.columns:
        data["is_daylight"] = (data["SolarRad_W_m_2"] > 10).astype(int)

    return data


df = add_engineered_features(df, TARGET_COL)

# ============================================================
# FEATURE LIST
# ============================================================
ENGINEERED_COLS = [
    "hour",
    "minute",
    "dayofweek",
    "dayofyear",
    "month",
    "is_weekend",

    "sin_day_1", "cos_day_1",
    "sin_day_2", "cos_day_2",
    "sin_day_3", "cos_day_3",

    "sin_week_1", "cos_week_1",
    "sin_week_2", "cos_week_2",

    f"{TARGET_COL}_lag_1",
    f"{TARGET_COL}_lag_2",
    f"{TARGET_COL}_lag_3",
    f"{TARGET_COL}_lag_6",
    f"{TARGET_COL}_lag_12",
    f"{TARGET_COL}_lag_24",
    f"{TARGET_COL}_lag_288",
    f"{TARGET_COL}_lag_576",
    f"{TARGET_COL}_lag_2016",

    f"{TARGET_COL}_roll_mean_3",
    f"{TARGET_COL}_roll_std_3",
    f"{TARGET_COL}_roll_mean_12",
    f"{TARGET_COL}_roll_std_12",
    f"{TARGET_COL}_roll_mean_288",
    f"{TARGET_COL}_roll_std_288",
    f"{TARGET_COL}_roll_mean_2016",
    f"{TARGET_COL}_roll_std_2016",

    "temp_sq",
    "heating_degree_18",
    "cooling_degree_22",
    "is_daylight",
]

FEATURE_COLS = BASE_WEATHER_COLS + ENGINEERED_COLS

# remove rows where engineered lags are unavailable
df = df.dropna(subset=[TARGET_COL] + FEATURE_COLS).reset_index(drop=True)

print("Final feature columns:")
print(FEATURE_COLS)
print("Number of features:", len(FEATURE_COLS))
print("Data range:", df["timestamp"].min(), "to", df["timestamp"].max())

# ============================================================
# LOAD FORECAST DATES
# ============================================================
with open(DATES_PATH, "r") as f:
    forecast_dates = json.load(f)

# ============================================================
# LOAD CHRONOS
# ============================================================
pipeline = Chronos2Pipeline.from_pretrained(
    "amazon/chronos-2",
    device_map="cpu",
)

# ============================================================
# FORECAST FUNCTION
# ============================================================
def run_one_forecast(season, day_name, test_start):

    test_start = pd.Timestamp(test_start)
    test_end = test_start + pd.Timedelta(days=1)

    history_df = df[df["timestamp"] < test_start].tail(CONTEXT_LEN).copy()

    future_df = df[
        (df["timestamp"] >= test_start)
        & (df["timestamp"] < test_end)
    ].copy()

    if len(history_df) == 0:
        print(f"Skipping {season}-{day_name}: no history before {test_start}")
        return None

    if len(future_df) != PRED_LEN:
        print(f"Skipping {season}-{day_name}: expected {PRED_LEN}, got {len(future_df)}")
        return None

    chronos_df = history_df[["timestamp", TARGET_COL] + FEATURE_COLS].copy()
    chronos_df = chronos_df.rename(columns={TARGET_COL: "target"})
    chronos_df["item_id"] = "building_consumption"

    chronos_df = chronos_df[
        ["item_id", "timestamp", "target"] + FEATURE_COLS
    ]

    pred_df = pipeline.predict_df(
        df=chronos_df,
        prediction_length=PRED_LEN,
        quantile_levels=QUANTILES,
        id_column="item_id",
        timestamp_column="timestamp",
        target="target",
        context_length=CONTEXT_LEN,
        freq="5min",
    )

    if "predictions" in pred_df.columns:
        pred_values = pred_df["predictions"].values[:PRED_LEN]
    elif "0.5" in pred_df.columns:
        pred_values = pred_df["0.5"].values[:PRED_LEN]
    elif 0.5 in pred_df.columns:
        pred_values = pred_df[0.5].values[:PRED_LEN]
    else:
        raise ValueError(f"Cannot find prediction column. Columns are: {pred_df.columns.tolist()}")

    out = future_df[["timestamp", TARGET_COL]].copy()
    out = out.rename(columns={TARGET_COL: "actual"})

    out["prediction"] = pred_values
    out["season"] = season
    out["day"] = day_name

    return out

# ============================================================
# RUN ALL DAYS
# ============================================================
all_results = []

start_time = time.time()

for season, days in forecast_dates.items():
    for day_name, test_start in days.items():

        print("=" * 80)
        print(f"Running {season} - {day_name}: {test_start}")

        res = run_one_forecast(season, day_name, test_start)

        if res is not None:
            all_results.append(res)

            day_path = OUTPUT_DIR / f"chronos2_5min_AF_{season}_{day_name}.csv"
            res.to_csv(day_path, index=False)

            print(f"Saved: {day_path}")

results_df = pd.concat(all_results, ignore_index=True)

all_path = OUTPUT_DIR / "chronos2_5min_AF_predictions.csv"
results_df.to_csv(all_path, index=False)

print(f"Saved combined predictions: {all_path}")
print(f"Runtime: {(time.time() - start_time) / 60:.2f} minutes")

Final feature columns:
['Temp__C', 'Hum__', 'AvgWindSpeed_M_s', 'Rain_Mm', 'SolarRad_W_m_2', 'hour', 'minute', 'dayofweek', 'dayofyear', 'month', 'is_weekend', 'sin_day_1', 'cos_day_1', 'sin_day_2', 'cos_day_2', 'sin_day_3', 'cos_day_3', 'sin_week_1', 'cos_week_1', 'sin_week_2', 'cos_week_2', 'avg_power_lag_1', 'avg_power_lag_2', 'avg_power_lag_3', 'avg_power_lag_6', 'avg_power_lag_12', 'avg_power_lag_24', 'avg_power_lag_288', 'avg_power_lag_576', 'avg_power_lag_2016', 'avg_power_roll_mean_3', 'avg_power_roll_std_3', 'avg_power_roll_mean_12', 'avg_power_roll_std_12', 'avg_power_roll_mean_288', 'avg_power_roll_std_288', 'avg_power_roll_mean_2016', 'avg_power_roll_std_2016', 'temp_sq', 'heating_degree_18', 'cooling_degree_22', 'is_daylight']
Number of features: 42
Data range: 2025-01-08 00:00:00 to 2025-12-31 23:55:00


Loading weights: 100%|██████████| 170/170 [00:00<00:00, 5088.83it/s]
c:\Users\Dimitrios\anaconda3\envs\chronos\Lib\site-packages\chronos\chronos2\pipeline.py:613: UserWarning: The specified context_length 8640 is greater than the model's default context length 8192. Resetting context_length to 8192.
  warnings.warn(


Running winter - day1: 2025-01-31 00:00:00
Saved: C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\chronos_5min_engineered\chronos2_5min_AF_winter_day1.csv
Running winter - day2: 2025-02-15 00:00:00


c:\Users\Dimitrios\anaconda3\envs\chronos\Lib\site-packages\chronos\chronos2\pipeline.py:613: UserWarning: The specified context_length 8640 is greater than the model's default context length 8192. Resetting context_length to 8192.
  warnings.warn(


Saved: C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\chronos_5min_engineered\chronos2_5min_AF_winter_day2.csv
Running winter - day3: 2025-12-02 00:00:00


...

c:\Users\Dimitrios\anaconda3\envs\chronos\Lib\site-packages\chronos\chronos2\pipeline.py:613: UserWarning: The specified context_length 8640 is greater than the model's default context length 8192. Resetting context_length to 8192.
  warnings.warn(


Saved: C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\chronos_5min_engineered\chronos2_5min_AF_winter_day5.csv
Running summer - day1: 2025-06-01 00:00:00


c:\Users\Dimitrios\anaconda3\envs\chronos\Lib\site-packages\chronos\chronos2\pipeline.py:613: UserWarning: The specified context_length 8640 is greater than the model's default context length 8192. Resetting context_length to 8192.
  warnings.warn(


Saved: C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\chronos_5min_engineered\chronos2_5min_AF_summer_day1.csv
Running summer - day2: 2025-06-24 00:00:00


c:\Users\Dimitrios\anaconda3\envs\chronos\Lib\site-packages\chronos\chronos2\pipeline.py:613: UserWarning: The specified context_length 8640 is greater than the model's default context length 8192. Resetting context_length to 8192.
  warnings.warn(


# RFFS

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
import time

from chronos import Chronos2Pipeline
from sklearn.ensemble import RandomForestRegressor

# ============================================================
# PATHS
# ============================================================
LOAD_PATH = r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\inputs\clean\Consumption_dataset_1min_imputed.csv"
WEATHER_PATH = r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\inputs\clean\meteo_imputed.csv"
DATES_PATH = r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\inputs\clean\forecast_dates.json"

OUTPUT_DIR = Path(
    r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\chronos_5min_engineered_RF95"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# SETTINGS
# ============================================================
TARGET_COL = "avg_power"

BASE_WEATHER_COLS = [
    "Temp__C",
    "Hum__",
    "AvgWindSpeed_M_s",
    "Rain_Mm",
    "SolarRad_W_m_2",
]

PRED_LEN = 288
CONTEXT_LEN = 288 * 30
QUANTILES = [0.1, 0.5, 0.9]

RF_IMPORTANCE_COVERAGE = 0.95

# ============================================================
# LOAD DATA
# ============================================================
load_df = pd.read_csv(LOAD_PATH)
weather_df = pd.read_csv(WEATHER_PATH)

load_df.columns = load_df.columns.str.strip()
weather_df.columns = weather_df.columns.str.strip()

weather_df = weather_df.rename(columns={"Date_Time": "timestamp"})

load_df["timestamp"] = pd.to_datetime(load_df["timestamp"])
weather_df["timestamp"] = pd.to_datetime(weather_df["timestamp"])

load_df = load_df.set_index("timestamp")
weather_df = weather_df.set_index("timestamp")

load_5min = load_df.resample("5min").mean()
weather_5min = weather_df.resample("5min").mean()

df = load_5min.join(weather_5min, how="inner").reset_index()
df = df.sort_values("timestamp").reset_index(drop=True)

# ============================================================
# ENGINEERED FEATURES
# ============================================================
def add_engineered_features(data, target_col):
    data = data.copy()
    ts = data["timestamp"]

    data["hour"] = ts.dt.hour
    data["minute"] = ts.dt.minute
    data["dayofweek"] = ts.dt.dayofweek
    data["dayofyear"] = ts.dt.dayofyear
    data["month"] = ts.dt.month
    data["is_weekend"] = (data["dayofweek"] >= 5).astype(int)

    steps_per_day = 288
    steps_per_week = 288 * 7

    step_of_day = data["hour"] * 12 + data["minute"] // 5
    step_of_week = data["dayofweek"] * steps_per_day + step_of_day

    for k in [1, 2, 3]:
        data[f"sin_day_{k}"] = np.sin(2 * np.pi * k * step_of_day / steps_per_day)
        data[f"cos_day_{k}"] = np.cos(2 * np.pi * k * step_of_day / steps_per_day)

    for k in [1, 2]:
        data[f"sin_week_{k}"] = np.sin(2 * np.pi * k * step_of_week / steps_per_week)
        data[f"cos_week_{k}"] = np.cos(2 * np.pi * k * step_of_week / steps_per_week)

    lag_steps = [
        1, 2, 3, 6, 12, 24,
        288,
        288 * 2,
        288 * 7,
    ]

    for lag in lag_steps:
        data[f"{target_col}_lag_{lag}"] = data[target_col].shift(lag)

    rolling_windows = [3, 12, 288, 288 * 7]

    for window in rolling_windows:
        shifted = data[target_col].shift(1)
        data[f"{target_col}_roll_mean_{window}"] = shifted.rolling(window).mean()
        data[f"{target_col}_roll_std_{window}"] = shifted.rolling(window).std()

    data["temp_sq"] = data["Temp__C"] ** 2
    data["heating_degree_18"] = np.maximum(18 - data["Temp__C"], 0)
    data["cooling_degree_22"] = np.maximum(data["Temp__C"] - 22, 0)
    data["is_daylight"] = (data["SolarRad_W_m_2"] > 10).astype(int)

    return data


df = add_engineered_features(df, TARGET_COL)

ENGINEERED_COLS = [
    "hour", "minute", "dayofweek", "dayofyear", "month", "is_weekend",

    "sin_day_1", "cos_day_1",
    "sin_day_2", "cos_day_2",
    "sin_day_3", "cos_day_3",

    "sin_week_1", "cos_week_1",
    "sin_week_2", "cos_week_2",

    f"{TARGET_COL}_lag_1",
    f"{TARGET_COL}_lag_2",
    f"{TARGET_COL}_lag_3",
    f"{TARGET_COL}_lag_6",
    f"{TARGET_COL}_lag_12",
    f"{TARGET_COL}_lag_24",
    f"{TARGET_COL}_lag_288",
    f"{TARGET_COL}_lag_576",
    f"{TARGET_COL}_lag_2016",

    f"{TARGET_COL}_roll_mean_3",
    f"{TARGET_COL}_roll_std_3",
    f"{TARGET_COL}_roll_mean_12",
    f"{TARGET_COL}_roll_std_12",
    f"{TARGET_COL}_roll_mean_288",
    f"{TARGET_COL}_roll_std_288",
    f"{TARGET_COL}_roll_mean_2016",
    f"{TARGET_COL}_roll_std_2016",

    "temp_sq",
    "heating_degree_18",
    "cooling_degree_22",
    "is_daylight",
]

ALL_FEATURE_COLS = BASE_WEATHER_COLS + ENGINEERED_COLS

df = df.dropna(subset=[TARGET_COL] + ALL_FEATURE_COLS).reset_index(drop=True)

print("Total candidate features:", len(ALL_FEATURE_COLS))
print("Data range:", df["timestamp"].min(), "to", df["timestamp"].max())

# ============================================================
# FORECAST DATES
# ============================================================
with open(DATES_PATH, "r") as f:
    forecast_dates = json.load(f)

# ============================================================
# CHRONOS
# ============================================================
pipeline = Chronos2Pipeline.from_pretrained(
    "amazon/chronos-2",
    device_map="cpu",
)

# ============================================================
# RANDOM FOREST 95% IMPORTANCE SELECTION
# ============================================================
def select_rf_95_features(train_df, feature_cols, target_col, season, day_name):
    X = train_df[feature_cols].copy()
    y = train_df[target_col].copy()

    rf = RandomForestRegressor(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1,
    )

    rf.fit(X, y)

    importance_df = pd.DataFrame({
        "feature": feature_cols,
        "importance": rf.feature_importances_,
    })

    importance_df = importance_df.sort_values(
        "importance", ascending=False
    ).reset_index(drop=True)

    total_importance = importance_df["importance"].sum()

    if total_importance == 0:
        selected_features = feature_cols
        importance_df["cumulative_importance"] = 0.0
    else:
        importance_df["importance_share"] = (
            importance_df["importance"] / total_importance
        )

        importance_df["cumulative_importance"] = (
            importance_df["importance_share"].cumsum()
        )

        selected_features = importance_df.loc[
            importance_df["cumulative_importance"] <= RF_IMPORTANCE_COVERAGE,
            "feature"
        ].tolist()

        # include the first feature that crosses 95%
        if len(selected_features) < len(importance_df):
            crossing_feature = importance_df.iloc[len(selected_features)]["feature"]
            selected_features.append(crossing_feature)

    if len(selected_features) == 0:
        selected_features = importance_df.head(1)["feature"].tolist()

    importance_df["selected"] = importance_df["feature"].isin(selected_features)
    importance_df["season"] = season
    importance_df["day"] = day_name

    return selected_features, importance_df

# ============================================================
# FORECAST FUNCTION
# ============================================================
def run_one_forecast(season, day_name, test_start):

    test_start = pd.Timestamp(test_start)
    test_end = test_start + pd.Timedelta(days=1)

    train_for_selection_df = df[df["timestamp"] < test_start].copy()

    history_df = df[df["timestamp"] < test_start].tail(CONTEXT_LEN).copy()

    future_df = df[
        (df["timestamp"] >= test_start)
        & (df["timestamp"] < test_end)
    ].copy()

    if len(train_for_selection_df) == 0:
        print(f"Skipping {season}-{day_name}: no training data before {test_start}")
        return None, None, None

    if len(history_df) == 0:
        print(f"Skipping {season}-{day_name}: no history before {test_start}")
        return None, None, None

    if len(future_df) != PRED_LEN:
        print(f"Skipping {season}-{day_name}: expected {PRED_LEN}, got {len(future_df)}")
        return None, None, None

    selected_features, importance_df = select_rf_95_features(
        train_df=train_for_selection_df,
        feature_cols=ALL_FEATURE_COLS,
        target_col=TARGET_COL,
        season=season,
        day_name=day_name,
    )

    print(f"Selected {len(selected_features)} / {len(ALL_FEATURE_COLS)} features")
    print(selected_features)

    chronos_df = history_df[["timestamp", TARGET_COL] + selected_features].copy()
    chronos_df = chronos_df.rename(columns={TARGET_COL: "target"})
    chronos_df["item_id"] = "building_consumption"

    chronos_df = chronos_df[
        ["item_id", "timestamp", "target"] + selected_features
    ]

    pred_df = pipeline.predict_df(
        df=chronos_df,
        prediction_length=PRED_LEN,
        quantile_levels=QUANTILES,
        id_column="item_id",
        timestamp_column="timestamp",
        target="target",
        context_length=CONTEXT_LEN,
        freq="5min",
    )

    if "predictions" in pred_df.columns:
        pred_values = pred_df["predictions"].values[:PRED_LEN]
    elif "0.5" in pred_df.columns:
        pred_values = pred_df["0.5"].values[:PRED_LEN]
    elif 0.5 in pred_df.columns:
        pred_values = pred_df[0.5].values[:PRED_LEN]
    else:
        raise ValueError(f"Cannot find prediction column. Columns are: {pred_df.columns.tolist()}")

    out = future_df[["timestamp", TARGET_COL]].copy()
    out = out.rename(columns={TARGET_COL: "actual"})

    out["prediction"] = pred_values
    out["season"] = season
    out["day"] = day_name
    out["n_selected_features"] = len(selected_features)

    selected_features_df = pd.DataFrame({
        "season": season,
        "day": day_name,
        "selected_feature": selected_features,
    })

    return out, importance_df, selected_features_df

# ============================================================
# RUN ALL DAYS
# ============================================================
all_results = []
all_importances = []
all_selected_features = []

start_time = time.time()

for season, days in forecast_dates.items():
    for day_name, test_start in days.items():

        print("=" * 80)
        print(f"Running {season} - {day_name}: {test_start}")

        res, importance_df, selected_features_df = run_one_forecast(
            season, day_name, test_start
        )

        if res is not None:
            all_results.append(res)
            all_importances.append(importance_df)
            all_selected_features.append(selected_features_df)

            day_path = OUTPUT_DIR / f"chronos2_5min_RFFS_{season}_{day_name}.csv"
            res.to_csv(day_path, index=False)

            imp_path = OUTPUT_DIR / f"RF95_importance_{season}_{day_name}.csv"
            importance_df.to_csv(imp_path, index=False)

            selected_path = OUTPUT_DIR / f"RF95_selected_features_{season}_{day_name}.csv"
            selected_features_df.to_csv(selected_path, index=False)

            print(f"Saved prediction: {day_path}")
            print(f"Saved importance: {imp_path}")
            print(f"Saved selected features: {selected_path}")

# ============================================================
# SAVE COMBINED FILES
# ============================================================
results_df = pd.concat(all_results, ignore_index=True)
importance_all_df = pd.concat(all_importances, ignore_index=True)
selected_all_df = pd.concat(all_selected_features, ignore_index=True)

results_df.to_csv(
    OUTPUT_DIR / "chronos2_5min_RF95_ALL_predictions.csv",
    index=False
)

importance_all_df.to_csv(
    OUTPUT_DIR / "RF95_ALL_importances.csv",
    index=False
)

selected_all_df.to_csv(
    OUTPUT_DIR / "RF95_ALL_selected_features.csv",
    index=False
)

print("Done.")
print(f"Runtime: {(time.time() - start_time) / 60:.2f} minutes")
print(f"Output folder: {OUTPUT_DIR}")

Total candidate features: 42
Data range: 2025-01-08 00:00:00 to 2025-12-31 23:55:00


Loading weights: 100%|██████████| 170/170 [00:00<00:00, 2619.37it/s]


Running winter - day1: 2025-01-31 00:00:00
Selected 14 / 42 features
['avg_power_lag_1', 'avg_power_roll_mean_3', 'avg_power_lag_2', 'avg_power_roll_mean_12', 'avg_power_lag_2016', 'avg_power_roll_mean_288', 'avg_power_lag_3', 'avg_power_lag_6', 'sin_day_1', 'avg_power_lag_288', 'avg_power_lag_576', 'avg_power_roll_std_12', 'avg_power_roll_std_3', 'avg_power_roll_mean_2016']


c:\Users\Dimitrios\anaconda3\envs\chronos\Lib\site-packages\chronos\chronos2\pipeline.py:613: UserWarning: The specified context_length 8640 is greater than the model's default context length 8192. Resetting context_length to 8192.
  warnings.warn(


Saved prediction: C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\chronos_5min_engineered_RF95\chronos2_5min_RFFS_winter_day1.csv
Saved importance: C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\chronos_5min_engineered_RF95\RF95_importance_winter_day1.csv
Saved selected features: C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\chronos_5min_engineered_RF95\RF95_selected_features_winter_day1.csv
Running winter - day2: 2025-02-15 00:00:00
Selected 16 / 42 features
['avg_power_lag_1', 'avg_power_lag_2', 'avg_power_roll_mean_3', 'avg_power_lag_2016', 'avg_power_roll_mean_12', 'avg_power_lag_288', 'avg_power_roll_std_12', 'avg_power_lag_6', 'avg_power_lag_576', 'sin_day_1', 'avg_power_lag_3', 'avg_power_lag_12', 'avg_power_roll_mean_288', 'sin_day_3', 'sin_day_

...

Saved prediction: C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\chronos_5min_engineered_RF95\chronos2_5min_RFFS_summer_day3.csv
Saved importance: C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\chronos_5min_engineered_RF95\RF95_importance_summer_day3.csv
Saved selected features: C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\chronos_5min_engineered_RF95\RF95_selected_features_summer_day3.csv
Running summer - day4: 2025-08-08 00:00:00
Selected 21 / 42 features
['avg_power_lag_1', 'avg_power_lag_2', 'avg_power_roll_mean_12', 'avg_power_lag_2016', 'sin_day_1', 'avg_power_roll_mean_3', 'avg_power_roll_std_12', 'avg_power_lag_6', 'avg_power_lag_3', 'avg_power_lag_288', 'avg_power_lag_576', 'avg_power_lag_12', 'avg_power_roll_std_3', 'avg_power_roll_mean_288'

c:\Users\Dimitrios\anaconda3\envs\chronos\Lib\site-packages\chronos\chronos2\pipeline.py:613: UserWarning: The specified context_length 8640 is greater than the model's default context length 8192. Resetting context_length to 8192.
  warnings.warn(


Saved prediction: C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\chronos_5min_engineered_RF95\chronos2_5min_RFFS_summer_day4.csv
Saved importance: C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\chronos_5min_engineered_RF95\RF95_importance_summer_day4.csv
Saved selected features: C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\chronos_5min_engineered_RF95\RF95_selected_features_summer_day4.csv
Running summer - day5: 2025-08-31 00:00:00
Selected 21 / 42 features
['avg_power_lag_1', 'avg_power_lag_2', 'avg_power_roll_mean_12', 'avg_power_lag_2016', 'avg_power_roll_mean_3', 'sin_day_1', 'avg_power_roll_std_12', 'avg_power_lag_576', 'avg_power_lag_288', 'avg_power_lag_6', 'avg_power_lag_3', 'avg_power_roll_std_3', 'avg_power_lag_12', 'avg_power_lag_24', 'avg_

c:\Users\Dimitrios\anaconda3\envs\chronos\Lib\site-packages\chronos\chronos2\pipeline.py:613: UserWarning: The specified context_length 8640 is greater than the model's default context length 8192. Resetting context_length to 8192.
  warnings.warn(


Saved prediction: C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\chronos_5min_engineered_RF95\chronos2_5min_RFFS_summer_day5.csv
Saved importance: C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\chronos_5min_engineered_RF95\RF95_importance_summer_day5.csv
Saved selected features: C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\chronos_5min_engineered_RF95\RF95_selected_features_summer_day5.csv
Done.
Runtime: 42.65 minutes
Output folder: C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\chronos_5min_engineered_RF95


# FSWL

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
import time

from chronos import Chronos2Pipeline
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV

# ============================================================
# PATHS
# ============================================================
LOAD_PATH = r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\inputs\clean\Consumption_dataset_1min_imputed.csv"
WEATHER_PATH = r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\inputs\clean\meteo_imputed.csv"
DATES_PATH = r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\inputs\clean\forecast_dates.json"

OUTPUT_DIR = Path(
    r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\chronos_5min_engineered_LASSO"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# SETTINGS
# ============================================================
TARGET_COL = "avg_power"

BASE_WEATHER_COLS = [
    "Temp__C",
    "Hum__",
    "AvgWindSpeed_M_s",
    "Rain_Mm",
    "SolarRad_W_m_2",
]

PRED_LEN = 288
CONTEXT_LEN = 288 * 30
QUANTILES = [0.1, 0.5, 0.9]

LASSO_COEF_THRESHOLD = 1e-6

# ============================================================
# LOAD DATA
# ============================================================
load_df = pd.read_csv(LOAD_PATH)
weather_df = pd.read_csv(WEATHER_PATH)

load_df.columns = load_df.columns.str.strip()
weather_df.columns = weather_df.columns.str.strip()

weather_df = weather_df.rename(columns={"Date_Time": "timestamp"})

load_df["timestamp"] = pd.to_datetime(load_df["timestamp"])
weather_df["timestamp"] = pd.to_datetime(weather_df["timestamp"])

load_df = load_df.set_index("timestamp")
weather_df = weather_df.set_index("timestamp")

load_5min = load_df.resample("5min").mean()
weather_5min = weather_df.resample("5min").mean()

df = load_5min.join(weather_5min, how="inner").reset_index()
df = df.sort_values("timestamp").reset_index(drop=True)

# ============================================================
# ENGINEERED FEATURES
# ============================================================
def add_engineered_features(data, target_col):
    data = data.copy()
    ts = data["timestamp"]

    data["hour"] = ts.dt.hour
    data["minute"] = ts.dt.minute
    data["dayofweek"] = ts.dt.dayofweek
    data["dayofyear"] = ts.dt.dayofyear
    data["month"] = ts.dt.month
    data["is_weekend"] = (data["dayofweek"] >= 5).astype(int)

    steps_per_day = 288
    steps_per_week = 288 * 7

    step_of_day = data["hour"] * 12 + data["minute"] // 5
    step_of_week = data["dayofweek"] * steps_per_day + step_of_day

    for k in [1, 2, 3]:
        data[f"sin_day_{k}"] = np.sin(2 * np.pi * k * step_of_day / steps_per_day)
        data[f"cos_day_{k}"] = np.cos(2 * np.pi * k * step_of_day / steps_per_day)

    for k in [1, 2]:
        data[f"sin_week_{k}"] = np.sin(2 * np.pi * k * step_of_week / steps_per_week)
        data[f"cos_week_{k}"] = np.cos(2 * np.pi * k * step_of_week / steps_per_week)

    lag_steps = [1, 2, 3, 6, 12, 24, 288, 576, 2016]

    for lag in lag_steps:
        data[f"{target_col}_lag_{lag}"] = data[target_col].shift(lag)

    rolling_windows = [3, 12, 288, 2016]

    for window in rolling_windows:
        shifted = data[target_col].shift(1)
        data[f"{target_col}_roll_mean_{window}"] = shifted.rolling(window).mean()
        data[f"{target_col}_roll_std_{window}"] = shifted.rolling(window).std()

    data["temp_sq"] = data["Temp__C"] ** 2
    data["heating_degree_18"] = np.maximum(18 - data["Temp__C"], 0)
    data["cooling_degree_22"] = np.maximum(data["Temp__C"] - 22, 0)
    data["is_daylight"] = (data["SolarRad_W_m_2"] > 10).astype(int)

    return data


df = add_engineered_features(df, TARGET_COL)

ENGINEERED_COLS = [
    "hour", "minute", "dayofweek", "dayofyear", "month", "is_weekend",

    "sin_day_1", "cos_day_1",
    "sin_day_2", "cos_day_2",
    "sin_day_3", "cos_day_3",

    "sin_week_1", "cos_week_1",
    "sin_week_2", "cos_week_2",

    f"{TARGET_COL}_lag_1",
    f"{TARGET_COL}_lag_2",
    f"{TARGET_COL}_lag_3",
    f"{TARGET_COL}_lag_6",
    f"{TARGET_COL}_lag_12",
    f"{TARGET_COL}_lag_24",
    f"{TARGET_COL}_lag_288",
    f"{TARGET_COL}_lag_576",
    f"{TARGET_COL}_lag_2016",

    f"{TARGET_COL}_roll_mean_3",
    f"{TARGET_COL}_roll_std_3",
    f"{TARGET_COL}_roll_mean_12",
    f"{TARGET_COL}_roll_std_12",
    f"{TARGET_COL}_roll_mean_288",
    f"{TARGET_COL}_roll_std_288",
    f"{TARGET_COL}_roll_mean_2016",
    f"{TARGET_COL}_roll_std_2016",

    "temp_sq",
    "heating_degree_18",
    "cooling_degree_22",
    "is_daylight",
]

ALL_FEATURE_COLS = BASE_WEATHER_COLS + ENGINEERED_COLS

df = df.dropna(subset=[TARGET_COL] + ALL_FEATURE_COLS).reset_index(drop=True)

print("Total candidate features:", len(ALL_FEATURE_COLS))
print("Data range:", df["timestamp"].min(), "to", df["timestamp"].max())

# ============================================================
# FORECAST DATES
# ============================================================
with open(DATES_PATH, "r") as f:
    forecast_dates = json.load(f)

# ============================================================
# CHRONOS
# ============================================================
pipeline = Chronos2Pipeline.from_pretrained(
    "amazon/chronos-2",
    device_map="cpu",
)

# ============================================================
# LASSO FEATURE SELECTION
# ============================================================
def select_lasso_features(train_df, feature_cols, target_col, season, day_name):
    X = train_df[feature_cols].copy()
    y = train_df[target_col].copy()

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("lasso", LassoCV(
            alphas=np.logspace(-4, 2, 40),
            cv=5,
            max_iter=20000,
            random_state=42,
            n_jobs=-1,
        ))
    ])

    model.fit(X, y)

    lasso = model.named_steps["lasso"]
    coefs = lasso.coef_

    coef_df = pd.DataFrame({
        "feature": feature_cols,
        "coefficient": coefs,
        "abs_coefficient": np.abs(coefs),
    })

    coef_df = coef_df.sort_values(
        "abs_coefficient", ascending=False
    ).reset_index(drop=True)

    selected_features = coef_df.loc[
        coef_df["abs_coefficient"] > LASSO_COEF_THRESHOLD,
        "feature"
    ].tolist()

    if len(selected_features) == 0:
        selected_features = coef_df.head(1)["feature"].tolist()

    coef_df["selected"] = coef_df["feature"].isin(selected_features)
    coef_df["season"] = season
    coef_df["day"] = day_name
    coef_df["lasso_alpha"] = lasso.alpha_

    return selected_features, coef_df

# ============================================================
# FORECAST FUNCTION
# ============================================================
def run_one_forecast(season, day_name, test_start):

    test_start = pd.Timestamp(test_start)
    test_end = test_start + pd.Timedelta(days=1)

    train_for_selection_df = df[df["timestamp"] < test_start].copy()

    history_df = df[df["timestamp"] < test_start].tail(CONTEXT_LEN).copy()

    future_df = df[
        (df["timestamp"] >= test_start)
        & (df["timestamp"] < test_end)
    ].copy()

    if len(train_for_selection_df) == 0:
        print(f"Skipping {season}-{day_name}: no training data before {test_start}")
        return None, None, None

    if len(history_df) == 0:
        print(f"Skipping {season}-{day_name}: no history before {test_start}")
        return None, None, None

    if len(future_df) != PRED_LEN:
        print(f"Skipping {season}-{day_name}: expected {PRED_LEN}, got {len(future_df)}")
        return None, None, None

    selected_features, coef_df = select_lasso_features(
        train_df=train_for_selection_df,
        feature_cols=ALL_FEATURE_COLS,
        target_col=TARGET_COL,
        season=season,
        day_name=day_name,
    )

    print(f"Selected {len(selected_features)} / {len(ALL_FEATURE_COLS)} features")
    print(selected_features)

    chronos_df = history_df[["timestamp", TARGET_COL] + selected_features].copy()
    chronos_df = chronos_df.rename(columns={TARGET_COL: "target"})
    chronos_df["item_id"] = "building_consumption"

    chronos_df = chronos_df[
        ["item_id", "timestamp", "target"] + selected_features
    ]

    pred_df = pipeline.predict_df(
        df=chronos_df,
        prediction_length=PRED_LEN,
        quantile_levels=QUANTILES,
        id_column="item_id",
        timestamp_column="timestamp",
        target="target",
        context_length=CONTEXT_LEN,
        freq="5min",
    )

    if "predictions" in pred_df.columns:
        pred_values = pred_df["predictions"].values[:PRED_LEN]
    elif "0.5" in pred_df.columns:
        pred_values = pred_df["0.5"].values[:PRED_LEN]
    elif 0.5 in pred_df.columns:
        pred_values = pred_df[0.5].values[:PRED_LEN]
    else:
        raise ValueError(f"Cannot find prediction column. Columns are: {pred_df.columns.tolist()}")

    out = future_df[["timestamp", TARGET_COL]].copy()
    out = out.rename(columns={TARGET_COL: "actual"})

    out["prediction"] = pred_values
    out["season"] = season
    out["day"] = day_name
    out["n_selected_features"] = len(selected_features)

    selected_features_df = pd.DataFrame({
        "season": season,
        "day": day_name,
        "selected_feature": selected_features,
    })

    return out, coef_df, selected_features_df

# ============================================================
# RUN ALL DAYS
# ============================================================
all_results = []
all_lasso_coefs = []
all_selected_features = []

start_time = time.time()

for season, days in forecast_dates.items():
    for day_name, test_start in days.items():

        print("=" * 80)
        print(f"Running {season} - {day_name}: {test_start}")

        res, coef_df, selected_features_df = run_one_forecast(
            season, day_name, test_start
        )

        if res is not None:
            all_results.append(res)
            all_lasso_coefs.append(coef_df)
            all_selected_features.append(selected_features_df)

            day_path = OUTPUT_DIR / f"chronos2_5min_FSWL_{season}_{day_name}.csv"
            res.to_csv(day_path, index=False)

            coef_path = OUTPUT_DIR / f"LASSO_coefficients_{season}_{day_name}.csv"
            coef_df.to_csv(coef_path, index=False)

            selected_path = OUTPUT_DIR / f"LASSO_selected_features_{season}_{day_name}.csv"
            selected_features_df.to_csv(selected_path, index=False)

            print(f"Saved prediction: {day_path}")
            print(f"Saved coefficients: {coef_path}")
            print(f"Saved selected features: {selected_path}")

# ============================================================
# SAVE COMBINED FILES
# ============================================================
results_df = pd.concat(all_results, ignore_index=True)
lasso_coefs_all_df = pd.concat(all_lasso_coefs, ignore_index=True)
selected_all_df = pd.concat(all_selected_features, ignore_index=True)

results_df.to_csv(
    OUTPUT_DIR / "chronos2_5min_LASSO_ALL_predictions.csv",
    index=False
)

lasso_coefs_all_df.to_csv(
    OUTPUT_DIR / "LASSO_ALL_coefficients.csv",
    index=False
)

selected_all_df.to_csv(
    OUTPUT_DIR / "LASSO_ALL_selected_features.csv",
    index=False
)

print("Done.")
print(f"Runtime: {(time.time() - start_time) / 60:.2f} minutes")
print(f"Output folder: {OUTPUT_DIR}")

Total candidate features: 42
Data range: 2025-01-08 00:00:00 to 2025-12-31 23:55:00


Loading weights: 100%|██████████| 170/170 [00:00<00:00, 2883.62it/s]


Running winter - day1: 2025-01-31 00:00:00
Selected 30 / 42 features
['avg_power_lag_1', 'avg_power_lag_2', 'avg_power_lag_3', 'avg_power_roll_mean_12', 'cos_day_3', 'avg_power_roll_std_12', 'avg_power_roll_std_3', 'avg_power_lag_24', 'avg_power_lag_576', 'cos_day_1', 'avg_power_lag_12', 'avg_power_lag_288', 'avg_power_lag_2016', 'cos_day_2', 'sin_week_2', 'avg_power_roll_std_2016', 'sin_day_2', 'cos_week_1', 'sin_day_3', 'avg_power_roll_std_288', 'cos_week_2', 'avg_power_roll_mean_2016', 'Hum__', 'is_daylight', 'sin_day_1', 'hour', 'Rain_Mm', 'avg_power_roll_mean_288', 'sin_week_1', 'dayofyear']


c:\Users\Dimitrios\anaconda3\envs\chronos\Lib\site-packages\chronos\chronos2\pipeline.py:613: UserWarning: The specified context_length 8640 is greater than the model's default context length 8192. Resetting context_length to 8192.
  warnings.warn(


Saved prediction: C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\chronos_5min_engineered_LASSO\chronos2_5min_FSWL_winter_day1.csv
Saved coefficients: C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\chronos_5min_engineered_LASSO\LASSO_coefficients_winter_day1.csv
Saved selected features: C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\chronos_5min_engineered_LASSO\LASSO_selected_features_winter_day1.csv
Running winter - day2: 2025-02-15 00:00:00
Selected 28 / 42 features
['avg_power_lag_1', 'avg_power_lag_2', 'avg_power_lag_3', 'avg_power_roll_mean_12', 'avg_power_roll_std_12', 'avg_power_lag_2016', 'avg_power_lag_576', 'cos_day_3', 'avg_power_lag_288', 'avg_power_roll_std_3', 'avg_power_lag_24', 'avg_power_lag_12', 'avg_power_roll_std_2016', 'cos_day_1', 

...

Saved prediction: C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\chronos_5min_engineered_LASSO\chronos2_5min_FSWL_summer_day3.csv
Saved coefficients: C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\chronos_5min_engineered_LASSO\LASSO_coefficients_summer_day3.csv
Saved selected features: C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\chronos_5min_engineered_LASSO\LASSO_selected_features_summer_day3.csv
Running summer - day4: 2025-08-08 00:00:00
Selected 33 / 42 features
['avg_power_lag_1', 'avg_power_lag_2', 'avg_power_roll_mean_12', 'avg_power_lag_3', 'avg_power_lag_2016', 'avg_power_roll_std_12', 'avg_power_lag_288', 'avg_power_lag_576', 'cos_day_3', 'cos_day_1', 'avg_power_roll_std_3', 'avg_power_roll_std_2016', 'SolarRad_W_m_2', 'cos_day_2', 'avg_powe

c:\Users\Dimitrios\anaconda3\envs\chronos\Lib\site-packages\chronos\chronos2\pipeline.py:613: UserWarning: The specified context_length 8640 is greater than the model's default context length 8192. Resetting context_length to 8192.
  warnings.warn(


Saved prediction: C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\chronos_5min_engineered_LASSO\chronos2_5min_FSWL_summer_day4.csv
Saved coefficients: C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\chronos_5min_engineered_LASSO\LASSO_coefficients_summer_day4.csv
Saved selected features: C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\chronos_5min_engineered_LASSO\LASSO_selected_features_summer_day4.csv
Running summer - day5: 2025-08-31 00:00:00
Selected 37 / 42 features
['avg_power_lag_1', 'avg_power_lag_2', 'avg_power_roll_mean_12', 'avg_power_lag_3', 'avg_power_lag_2016', 'avg_power_roll_std_12', 'avg_power_lag_288', 'avg_power_lag_576', 'cos_day_3', 'cos_day_1', 'avg_power_roll_std_3', 'SolarRad_W_m_2', 'dayofyear', 'temp_sq', 'cos_day_2', 'avg_power_r

c:\Users\Dimitrios\anaconda3\envs\chronos\Lib\site-packages\chronos\chronos2\pipeline.py:613: UserWarning: The specified context_length 8640 is greater than the model's default context length 8192. Resetting context_length to 8192.
  warnings.warn(


Saved prediction: C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\chronos_5min_engineered_LASSO\chronos2_5min_FSWL_summer_day5.csv
Saved coefficients: C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\chronos_5min_engineered_LASSO\LASSO_coefficients_summer_day5.csv
Saved selected features: C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\chronos_5min_engineered_LASSO\LASSO_selected_features_summer_day5.csv
Done.
Runtime: 6.84 minutes
Output folder: C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\chronos_5min_engineered_LASSO


# Univariate

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
import time

from chronos import Chronos2Pipeline

# ============================================================
# PATHS
# ============================================================
LOAD_PATH = r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\inputs\clean\Consumption_dataset_1min_imputed.csv"
DATES_PATH = r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\inputs\clean\forecast_dates.json"

OUTPUT_DIR = Path(
    r"C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\chronos_5min_univariate"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# SETTINGS
# ============================================================
TARGET_COL = "avg_power"

PRED_LEN = 288
CONTEXT_LEN = 288 * 30
QUANTILES = [0.1, 0.5, 0.9]

# ============================================================
# LOAD LOAD DATA ONLY
# ============================================================
load_df = pd.read_csv(LOAD_PATH)
load_df.columns = load_df.columns.str.strip()

load_df["timestamp"] = pd.to_datetime(load_df["timestamp"])
load_df = load_df.set_index("timestamp")

# 1-min load -> 5-min
load_5min = load_df.resample("5min").mean()

df = load_5min.reset_index()
df = df.sort_values("timestamp").reset_index(drop=True)
df = df.dropna(subset=[TARGET_COL]).reset_index(drop=True)

print("Data range:", df["timestamp"].min(), "to", df["timestamp"].max())
print(df.head())

# ============================================================
# FORECAST DATES
# ============================================================
with open(DATES_PATH, "r") as f:
    forecast_dates = json.load(f)

# ============================================================
# LOAD CHRONOS
# ============================================================
pipeline = Chronos2Pipeline.from_pretrained(
    "amazon/chronos-2",
    device_map="cpu",
)

# ============================================================
# FORECAST FUNCTION
# ============================================================
def run_one_forecast(season, day_name, test_start):

    test_start = pd.Timestamp(test_start)
    test_end = test_start + pd.Timedelta(days=1)

    history_df = df[df["timestamp"] < test_start].tail(CONTEXT_LEN).copy()

    future_df = df[
        (df["timestamp"] >= test_start)
        & (df["timestamp"] < test_end)
    ].copy()

    if len(history_df) == 0:
        print(f"Skipping {season}-{day_name}: no history before {test_start}")
        return None

    if len(future_df) != PRED_LEN:
        print(f"Skipping {season}-{day_name}: expected {PRED_LEN}, got {len(future_df)}")
        return None

    # Univariate Chronos input: only item_id, timestamp, target
    chronos_df = history_df[["timestamp", TARGET_COL]].copy()
    chronos_df = chronos_df.rename(columns={TARGET_COL: "target"})
    chronos_df["item_id"] = "building_consumption"

    chronos_df = chronos_df[
        ["item_id", "timestamp", "target"]
    ]

    pred_df = pipeline.predict_df(
        df=chronos_df,
        prediction_length=PRED_LEN,
        quantile_levels=QUANTILES,
        id_column="item_id",
        timestamp_column="timestamp",
        target="target",
        context_length=CONTEXT_LEN,
        freq="5min",
    )

    if "predictions" in pred_df.columns:
        pred_values = pred_df["predictions"].values[:PRED_LEN]
    elif "0.5" in pred_df.columns:
        pred_values = pred_df["0.5"].values[:PRED_LEN]
    elif 0.5 in pred_df.columns:
        pred_values = pred_df[0.5].values[:PRED_LEN]
    else:
        raise ValueError(f"Cannot find prediction column. Columns are: {pred_df.columns.tolist()}")

    out = future_df[["timestamp", TARGET_COL]].copy()
    out = out.rename(columns={TARGET_COL: "actual"})

    out["prediction"] = pred_values
    out["season"] = season
    out["day"] = day_name

    return out

# ============================================================
# RUN ALL DAYS
# ============================================================
all_results = []

start_time = time.time()

for season, days in forecast_dates.items():
    for day_name, test_start in days.items():

        print("=" * 80)
        print(f"Running {season} - {day_name}: {test_start}")

        res = run_one_forecast(season, day_name, test_start)

        if res is not None:
            all_results.append(res)

            day_path = OUTPUT_DIR / f"chronos2_5min_univariate_{season}_{day_name}.csv"
            res.to_csv(day_path, index=False)

            print(f"Saved prediction: {day_path}")

# ============================================================
# SAVE COMBINED FILE
# ============================================================
results_df = pd.concat(all_results, ignore_index=True)

all_path = OUTPUT_DIR / "chronos2_5min_univariate_ALL_predictions.csv"
results_df.to_csv(all_path, index=False)

print("Done.")
print(f"Saved combined predictions: {all_path}")
print(f"Runtime: {(time.time() - start_time) / 60:.2f} minutes")

Data range: 2025-01-01 00:00:00 to 2025-12-31 23:55:00
            timestamp  avg_power
0 2025-01-01 00:00:00  2954.2428
1 2025-01-01 00:05:00  2642.1732
2 2025-01-01 00:10:00  2646.2412
3 2025-01-01 00:15:00  2685.5580
4 2025-01-01 00:20:00  2665.8096


Loading weights: 100%|██████████| 170/170 [00:00<00:00, 4809.53it/s]


Running winter - day1: 2025-01-31 00:00:00


c:\Users\Dimitrios\anaconda3\envs\chronos\Lib\site-packages\chronos\chronos2\pipeline.py:613: UserWarning: The specified context_length 8640 is greater than the model's default context length 8192. Resetting context_length to 8192.
  warnings.warn(


Saved prediction: C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\chronos_5min_univariate\chronos2_5min_univariate_winter_day1.csv
Running winter - day2: 2025-02-15 00:00:00


...

Saved prediction: C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\chronos_5min_univariate\chronos2_5min_univariate_summer_day3.csv
Running summer - day4: 2025-08-08 00:00:00


c:\Users\Dimitrios\anaconda3\envs\chronos\Lib\site-packages\chronos\chronos2\pipeline.py:613: UserWarning: The specified context_length 8640 is greater than the model's default context length 8192. Resetting context_length to 8192.
  warnings.warn(


Saved prediction: C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\chronos_5min_univariate\chronos2_5min_univariate_summer_day4.csv
Running summer - day5: 2025-08-31 00:00:00


c:\Users\Dimitrios\anaconda3\envs\chronos\Lib\site-packages\chronos\chronos2\pipeline.py:613: UserWarning: The specified context_length 8640 is greater than the model's default context length 8192. Resetting context_length to 8192.
  warnings.warn(


Saved prediction: C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\chronos_5min_univariate\chronos2_5min_univariate_summer_day5.csv
Done.
Saved combined predictions: C:\Users\Dimitrios\Desktop\Does-Feature-Selection-Still-Matter-for-Time-Series-Foundation-Models-in-Load-Forecasting-\outputs\chronos_5min_univariate\chronos2_5min_univariate_ALL_predictions.csv
Runtime: 0.16 minutes


# end